# 21 自编码器 Autoencoder

依赖安装说明：`pip install numpy matplotlib scikit-learn torch`

自编码器是一种把输入压缩到低维表示，再重建回输入的神经网络。它常用于降维、表示学习、去噪和异常检测入门。


## 0. 学习目标和阅读地图

自编码器的重点是“压缩再重建”。你需要掌握：

1. encoder、latent、decoder 各自作用。
2. bottleneck 为什么迫使模型学习压缩表示。
3. 重建误差如何用于降维或异常检测。
4. 自编码器和 PCA 的关系。


## 1. 数学逻辑

编码器把输入压缩成 latent 表示：

$$z = encoder(x)$$

解码器重建输入：

$$\hat x = decoder(z)$$

训练目标是让重建误差尽量小：

$$L = \frac{1}{n}\sum_i ||x_i-\hat x_i||^2$$

如果 latent 维度很小，模型必须学会保留最重要的信息。


## 1.1 推导拆开看：为什么 bottleneck 重要

如果 latent 维度和模型容量都很大，自编码器可能学会近似复制输入：

$$decoder(encoder(x)) \approx x$$

这不一定产生有用表示。bottleneck 限制 `z` 的维度，迫使模型保留最重要的信息。

线性自编码器在 MSE 下学到的子空间和 PCA 有密切关系；非线性自编码器则可以学习弯曲流形。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
t = np.linspace(0, 2*np.pi, 500)
X = np.column_stack([np.cos(t), np.sin(t)]) + np.random.normal(scale=0.08, size=(500, 2))
plt.scatter(X[:,0], X[:,1], s=12)
plt.axis('equal')
plt.title('二维圆环数据')
plt.show()


## 1.2 这个圆环例子想说明什么

圆环数据在二维空间里是弯曲结构。如果只用 1 维线性 PCA 重建，会把点投影到一条直线上。

非线性自编码器有机会用 1 维 latent 表示圆环上的位置，再通过 decoder 重建回二维曲线。


In [ ]:
# 从零直觉：线性自编码器和 PCA 有亲缘关系
# 如果 encoder/decoder 都是线性的，且使用 MSE，学到的低维空间类似 PCA 子空间。
Xc = X - X.mean(axis=0)
u, s, vt = np.linalg.svd(Xc, full_matrices=False)
Z = Xc @ vt[:1].T
X_recon = Z @ vt[:1] + X.mean(axis=0)
print('用 1 维 PCA 重建的 MSE:', round(np.mean((X - X_recon) ** 2), 4))

plt.scatter(X[:,0], X[:,1], s=10, alpha=0.4, label='original')
plt.scatter(X_recon[:,0], X_recon[:,1], s=10, alpha=0.4, label='1D reconstruction')
plt.legend()
plt.axis('equal')
plt.show()


## 1.3 从零直觉代码怎么读

PCA 重建展示了线性压缩的限制：

1. `Z = Xc @ vt[:1].T`：把二维点压成 1 维。
2. `X_recon = Z @ vt[:1] + mean`：从 1 维还原回二维。
3. MSE 衡量重建损失。

这为理解非线性 autoencoder 做铺垫。


In [ ]:
# PyTorch 实战：非线性 autoencoder
import torch
from torch import nn

Xt = torch.tensor(X, dtype=torch.float32)

class AutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(2, 16), nn.Tanh(), nn.Linear(16, 1))
        self.decoder = nn.Sequential(nn.Linear(1, 16), nn.Tanh(), nn.Linear(16, 2))
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

model = AutoEncoder()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
loss_fn = nn.MSELoss()

for step in range(500):
    recon = model(Xt)
    loss = loss_fn(recon, Xt)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    recon = model(Xt).numpy()
print('autoencoder reconstruction MSE:', round(np.mean((X - recon) ** 2), 4))

plt.scatter(X[:,0], X[:,1], s=10, alpha=0.35, label='original')
plt.scatter(recon[:,0], recon[:,1], s=10, alpha=0.35, label='reconstruction')
plt.legend()
plt.axis('equal')
plt.title('非线性自编码器重建')
plt.show()


In [ ]:
# 诊断：重建误差分布，可作为异常检测的基础
recon_error = np.mean((X - recon) ** 2, axis=1)
plt.hist(recon_error, bins=30)
plt.title('每个样本的重建误差分布')
plt.xlabel('reconstruction error')
plt.ylabel('count')
plt.show()
print('95% 分位数阈值:', round(np.quantile(recon_error, 0.95), 5))


## 2.1 如何诊断自编码器

重建误差低说明模型能复原输入，但不一定说明 latent 表示对下游任务有用。

用于异常检测时，常见做法是只用正常样本训练。测试时，如果某个样本重建误差很高，就可能是异常。


## 2. 常见误区

- 如果 latent 维度太大，自编码器可能只是复制输入，学不到有用压缩。
- 重建误差低不代表表示一定适合分类。
- 用于异常检测时，要确认训练集主要是正常样本。

## 3. 小实验

- 把 latent 维度从 1 改成 2。
- 增加噪声，做去噪自编码器。
- 对比 PCA 重建和非线性自编码器重建。


## 5. 复习清单

- 自编码器由 encoder、latent、decoder 组成。
- bottleneck 决定压缩强度。
- 线性自编码器和 PCA 有关系。
- 重建误差可以用于异常检测，但阈值需要验证。
